# Target Problem

The project proposes to use a deep reinforcement learning framework to learn a profitable stock trading strategy, with the goal to optimize the cumulative return and Alpha. It would select S&P500 Index with the top 20 market capitalization stocks as our trading stock pool. The input to the algorithm is the market trend for these stocks in the last month, remaining balance, and current portfolio. The model agent output is a series of trading actions among stocks. The available trading
action options are: sell, buy and hold. The market data in the most recent months will be used to feed
the model performance evaluation.

The algorithm is trained using Deep Reinforcement Learning (DRL) algorithms and the components of the reinforcement learning environment are:

* Action: The action space describes the allowed actions that the agent interacts with the
environment. Normally, a ∈ A includes three actions: a ∈ {−1, 0, 1}, where −1, 0, 1 represent
selling, holding, and buying one stock. Also, an action can be carried upon multiple shares. We use
an action space {−k, ..., −1, 0, 1, ..., k}, where k denotes the number of shares. For example, "Buy
10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or −10, respectively

* Reward function: r(s, a, s′) is the incentive mechanism for an agent to learn a better action. The change of the portfolio value when action a is taken at state s and arriving at new state s',  i.e., r(s, a, s′) = v′ − v, where v′ and v represent the portfolio
values at state s′ and s, respectively

* State: The state space describes the observations that the agent receives from the environment. Just as a human trader needs to analyze various information before executing a trade, so
our trading agent observes many different features to better learn in an interactive environment.

* Environment: SP500 top 20 companies


The data of the single stock that we will be using for this case study is obtained from Yahoo Finance API. The data contains Open-High-Low-Close price and volume.


## Environment Setup

### Import Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from datetime import datetime
from datetime import timedelta

%matplotlib inline
from rl.config import config
from rl.marketdata.yahoodownloader import YahooDownloader
from rl.preprocessing.preprocessors import FeatureEngineer
from rl.preprocessing.data import data_split
from rl.environment.env_stocktrading import StockTradingEnv
from rl.model.models import DRLAgent
from rl.trade.backtest import backtest_stats, backtest_plot, get_daily_return, get_baseline

from pprint import pprint

import itertools

/Users/minchen.wang/Documents/financial-agent/.venv/lib/python3.12/site-packages/pyfolio/pos.py:25: UserWarning: Module "zipline.assets" not found; multipliers will not be applied to position notionals.
  warnings.warn(


### Cache Folders

In [2]:
import os
if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

### Fetch Data

In [3]:
# from config.py start_date is a string
start_date = config.START_DATE
# from config.py end_date is a string
end_date = config.END_DATE
# from config.py split_date is a string
split_date = config.SPLIT_DATE
# config target ticker
target_ticker = config.SP500_20_TICKER
# config tech_indicator_list
tech_indicator_list = config.TECHNICAL_INDICATORS_LIST
print("training period: {}-{}, testing period: {}-{}\n".format(start_date, split_date, split_date, end_date))
print("target ticker list:\n {}\n".format(target_ticker))
print("tech_indicator_list:\n {}\n".format(tech_indicator_list))

training period: 2000-01-01-2023-10-01, testing period: 2023-10-01-2025-10-01

target ticker list:
 ['AAPL', 'MSFT', 'AMZN', 'BRK-B', 'JPM', 'JNJ', 'UNH', 'HD', 'PG', 'NVDA', 'DIS', 'BAC', 'CMCSA', 'XOM', 'VZ', 'T', 'ADBE', 'PFE', 'CSCO', 'INTC']

tech_indicator_list:
 ['macd', 'boll_ub', 'boll_lb', 'rsi_10', 'rsi_20', 'cci_10', 'cci_20', 'dx_30', 'close_20_sma', 'close_60_sma', 'close_120_sma', 'close_20_ema', 'close_60_ema', 'close_120_ema']



In [4]:
df = YahooDownloader(start_date = start_date,
                     end_date = end_date,
                     ticker_list = target_ticker).fetch_data()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (129500, 8)


In [5]:
df.shape

(129500, 8)

In [6]:
df.sort_values(['date','tic'],ignore_index=True).head()

,date,close,high,low,open,volume,tic,day
0,2000-01-03,0.840094,1.004464,0.907924,0.936384,535796800,AAPL,0
1,2000-01-03,16.274672,16.875000,16.062500,16.812500,7384400,ADBE,0
2,2000-01-03,4.468750,4.478125,3.952344,4.075000,322352000,AMZN,0
3,2000-01-03,12.490809,25.125000,24.000000,25.125000,13705800,BAC,0
4,2000-01-03,35.299999,36.580002,34.820000,36.500000,875000,BRK-B,0


## Data Preprocessing
Data preprocessing is a crucial step for training a high quality machine learning model. We need to check for missing data and do feature engineering in order to convert the data into a model-ready state.
* Add technical indicators. In practical trading, various information needs to be taken into account, for example the historical stock prices, current holding shares, technical indicators, etc. In this article, we demonstrate two trend-following technical indicators: MACD and RSI.
* Add turbulence index. Risk-aversion reflects whether an investor will choose to preserve the capital. It also influences one's trading strategy when facing different market volatility level. To control the risk in a worst-case scenario, such as financial crisis of 2007–2008, FinRL employs the financial turbulence index that measures extreme asset price fluctuation.

### Feature Engineering

In [7]:
fe = FeatureEngineer(
                    use_technical_indicator=True,
                    tech_indicator_list = tech_indicator_list,
                    use_turbulence=True,
                    user_defined_feature = False)

processed = fe.preprocess_data(df)

Successfully added technical indicators
Successfully added turbulence index


/Users/minchen.wang/Documents/financial-agent/src/core_trading/rl/preprocessing/preprocessors.py:62: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="bfill").fillna(method="ffill")


In [8]:
print(processed.shape)
processed.head()

(129500, 23)


,date,close,high,low,open,volume,tic,day,macd,boll_ub,...,cci_10,cci_20,dx_30,close_20_sma,close_60_sma,close_120_sma,close_20_ema,close_60_ema,close_120_ema,turbulence
0,2000-01-03,0.840094,1.004464,0.907924,0.936384,535796800,AAPL,0,0.0,0.904846,...,-66.666667,-66.666667,100.0,0.840094,0.840094,0.840094,0.840094,0.840094,0.840094,0.0
1,2000-01-03,16.274672,16.875000,16.062500,16.812500,7384400,ADBE,0,0.0,0.904846,...,-66.666667,-66.666667,100.0,16.274672,16.274672,16.274672,16.274672,16.274672,16.274672,0.0
2,2000-01-03,4.468750,4.478125,3.952344,4.075000,322352000,AMZN,0,0.0,0.904846,...,-66.666667,-66.666667,100.0,4.468750,4.468750,4.468750,4.468750,4.468750,4.468750,0.0
3,2000-01-03,12.490809,25.125000,24.000000,25.125000,13705800,BAC,0,0.0,0.904846,...,-66.666667,-66.666667,100.0,12.490809,12.490809,12.490809,12.490809,12.490809,12.490809,0.0
4,2000-01-03,35.299999,36.580002,34.820000,36.500000,875000,BRK-B,0,0.0,0.904846,...,-66.666667,-66.666667,100.0,35.299999,35.299999,35.299999,35.299999,35.299999,35.299999,0.0


### Data display

In [9]:
list_ticker = processed["tic"].unique().tolist()
print(list_ticker)
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))
processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])
processed_full = processed_full.fillna(0)

['AAPL', 'ADBE', 'AMZN', 'BAC', 'BRK-B', 'CMCSA', 'CSCO', 'DIS', 'HD', 'INTC', 'JNJ', 'JPM', 'MSFT', 'NVDA', 'PFE', 'PG', 'T', 'UNH', 'VZ', 'XOM']


In [10]:
processed_full.sort_values(['date','tic'],ignore_index=True).head(10)

,date,tic,close,high,low,open,volume,day,macd,boll_ub,...,cci_10,cci_20,dx_30,close_20_sma,close_60_sma,close_120_sma,close_20_ema,close_60_ema,close_120_ema,turbulence
0,2000-01-03,AAPL,0.840094,1.004464,0.907924,0.936384,535796800.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,0.840094,0.840094,0.840094,0.840094,0.840094,0.840094,0.0
1,2000-01-03,ADBE,16.274672,16.875000,16.062500,16.812500,7384400.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,16.274672,16.274672,16.274672,16.274672,16.274672,16.274672,0.0
2,2000-01-03,AMZN,4.468750,4.478125,3.952344,4.075000,322352000.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,4.468750,4.468750,4.468750,4.468750,4.468750,4.468750,0.0
3,2000-01-03,BAC,12.490809,25.125000,24.000000,25.125000,13705800.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,12.490809,12.490809,12.490809,12.490809,12.490809,12.490809,0.0
4,2000-01-03,BRK-B,35.299999,36.580002,34.820000,36.500000,875000.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,35.299999,35.299999,35.299999,35.299999,35.299999,35.299999,0.0
5,2000-01-03,CMCSA,10.698268,16.333332,15.062500,16.145832,2333700.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,10.698268,10.698268,10.698268,10.698268,10.698268,10.698268,0.0
6,2000-01-03,CSCO,35.148067,55.125000,51.781250,54.968750,53076000.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,35.148067,35.148067,35.148067,35.148067,35.148067,35.148067,0.0
7,2000-01-03,DIS,22.736223,29.533344,28.361876,28.855125,8402230.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,22.736223,22.736223,22.736223,22.736223,22.736223,22.736223,0.0
8,2000-01-03,HD,38.163876,69.187500,63.812500,68.625000,12030800.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,38.163876,38.163876,38.163876,38.163876,38.163876,38.163876,0.0
9,2000-01-03,INTC,24.710653,43.687500,41.625000,41.632812,57710200.0,0.0,0.0,0.904846,...,-66.666667,-66.666667,100.0,24.710653,24.710653,24.710653,24.710653,24.710653,24.710653,0.0


### Data Split

In [21]:
train = data_split(processed_full, start_date, end_date)
evaluate = data_split(processed_full, split_date, end_date)
print(train.shape)
print(evaluate.shape)

(129500, 23)
(10020, 23)


Rolling Predict Time Periods

In [22]:
predict_start_date=['2023-10-01', '2024-01-01', '2024-04-01', '2024-07-01', '2024-10-01']
predict_end_date=['2024-10-01', '2025-01-01', '2025-04-01', '2025-07-01', '2025-10-01']

## RL Environment
Considering the stochastic and interactive nature of the automated stock trading tasks, a financial task is modeled as a **Markov Decision Process (MDP)** problem. The training process involves observing stock price change, taking an action and reward's calculation to have the agent adjusting its strategy accordingly. By interacting with the environment, the trading agent will derive a trading strategy with the maximized rewards as time proceeds.

Our trading environments, based on OpenAI Gym framework, simulate live stock markets with real market data according to the principle of time-driven simulation.

The action space describes the allowed actions that the agent interacts with the environment. Normally, action a includes three actions: {-1, 0, 1}, where -1, 0, 1 represent selling, holding, and buying one share. Also, an action can be carried upon multiple shares. We use an action space {-k,…,-1, 0, 1, …, k}, where k denotes the number of shares to buy and -k denotes the number of shares to sell. For example, "Buy 10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or -10, respectively. The continuous action space needs to be normalized to [-1, 1], since the policy is defined on a Gaussian distribution, which needs to be normalized and symmetric.

In [23]:
stock_dimension = len(processed_full.tic.unique())
state_space = 1 + 2*stock_dimension + len(tech_indicator_list)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 20, State Space: 321


### Basic Model Train

In [24]:
def train_predict(model_name, tb_log_name, dataset, start, split, end, env_params, model_kwargs, total_timesteps):
    e_train_gym = StockTradingEnv(df = data_split(dataset, start, split), **env_params)
    e_evaluation_gym = StockTradingEnv(df=data_split(dataset, split, end), **env_params)
    
    env_train, _ = e_train_gym.get_sb_env()
    
    agent = DRLAgent(env = env_train)
    model = agent.get_model(model_name, model_kwargs = model_kwargs)
    
    trained_model = agent.train_model(
        model=model, 
        tb_log_name=tb_log_name,
        total_timesteps=total_timesteps)

    train_value, train_actions = DRLAgent.DRL_prediction(trained_model, e_evaluation_gym)
    test_value, test_actions = DRLAgent.DRL_prediction(trained_model, e_evaluation_gym)
    return train_value, train_actions, test_value, test_actions

In [25]:
def rolling_predict(model_name, tb_log_name, dataset, start, split, end, env_params, model_kwargs, total_timesteps):
    e_train_gym = StockTradingEnv(df = data_split(dataset, start, split), **env_params)
    e_evaluation_gym = StockTradingEnv(df=data_split(dataset, split, end), **env_params)
    
    env_train, _ = e_train_gym.get_sb_env()
    
    agent = DRLAgent(env = env_train)
    model = agent.get_model(model_name, model_kwargs = model_kwargs)
    
    trained_model = agent.train_model(
        model=model, 
        tb_log_name=tb_log_name,
        total_timesteps=total_timesteps)

    test_value, test_actions = DRLAgent.DRL_prediction(trained_model, e_evaluation_gym)
    return test_value, test_actions

In [26]:
def train_and_predict(model_name, dataset, env_params, model_kwargs, total_timesteps):
    start = start_date
    split=predict_start_date[0]
    end=predict_end_date[0]
    
    
    print("rolling predict: {}:{}:{}, env_params: {}".format(start, split, end, env_params))
    train_value, train_actions, test_value, test_actions = train_predict(model_name, "{}_train_{}".format(model_name, 0), dataset, start, split, end, env_params, model_kwargs, total_timesteps)
    
    rolling_env_params = env_params.copy()
    
    for i in range(1, len(predict_start_date)):
        rolling_env_params['initial_amount']=test_value.iloc[-1].account_value
        
        start = start_date
        split=predict_start_date[i]
        end=predict_end_date[i]
        
        print("rolling predict: {}:{}:{}, env_params: {}".format(start, split, end, rolling_env_params))
        
        value, actions = rolling_predict(model_name, "{}_rolling_{}".format(model_name, i), dataset, start, split, end, rolling_env_params, model_kwargs, total_timesteps)
        test_value=pd.concat([test_value, value])
        test_actions=pd.concat([test_actions, actions])
    return train_value, train_actions, test_value, test_actions

In [27]:
print(predict_start_date)
print(predict_end_date)

['2023-10-01', '2024-01-01', '2024-04-01', '2024-07-01', '2024-10-01']
['2024-10-01', '2025-01-01', '2025-04-01', '2025-07-01', '2025-10-01']


In [28]:
ENV_PARAMS = {
    "hmax": 100, 
    "initial_amount": 1000000, 
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space, 
    "stock_dim": stock_dimension, 
    "tech_indicator_list": tech_indicator_list, 
    "action_space": stock_dimension, 
    "reward_scaling": 1e-6
}
A2C_PARAMS = {
    "n_steps": 5, 
    "ent_coef": 0.005, 
    "learning_rate": 0.0001
}
PPO_PARAMS = {
    "n_steps": 2048,
    "ent_coef": 0.005,
    "learning_rate": 0.0001,
    "batch_size": 128,
}
DDPG_PARAMS = {
    "batch_size": 128, 
    "buffer_size": 100000, 
    "learning_rate": 0.0005
}
TD3_PARAMS = {
    "batch_size": 128, 
    "buffer_size": 100000, 
    "learning_rate": 0.0005
}
SAC_PARAMS = {
    "batch_size": 128,
    "buffer_size": 100000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1"
}
total_timesteps = 50000

In [29]:
df_value, df_actions = dict(), dict()

In [30]:
%%time
%%capture

df_value['train_a2c'],df_actions['train_a2c'],df_value['test_a2c'],df_actions['test_a2c']=train_and_predict("a2c", processed_full, ENV_PARAMS, A2C_PARAMS, total_timesteps=100000)

CPU times: user 58min 43s, sys: 14min 46s, total: 1h 13min 29s
Wall time: 1h 31min 14s


In [ ]:
%%time
%%capture

df_value['train_ppo'],df_actions['train_ppo'],df_value['test_ppo'],df_actions['test_ppo']=train_and_predict("ppo", processed_full, ENV_PARAMS, PPO_PARAMS, total_timesteps=100000)

CPU times: user 17min 35s, sys: 1min 28s, total: 19min 4s
Wall time: 18min 49s


In [ ]:
%%time
%%capture

df_value['train_ddpg'],df_actions['train_ddpg'],df_value['test_ddpg'],df_actions['test_ddpg']=train_and_predict("ddpg", processed_full, ENV_PARAMS, DDPG_PARAMS, total_timesteps=100000)

CPU times: user 33min 36s, sys: 28min 9s, total: 1h 1min 45s
Wall time: 3h 37min 4s


In [ ]:
%%time
%%capture

df_value['train_td3'],df_actions['train_td3'],df_value['test_td3'],df_actions['test_td3']=train_and_predict("td3", processed_full, ENV_PARAMS, TD3_PARAMS, total_timesteps=100000)

CPU times: user 34min 52s, sys: 31min 22s, total: 1h 6min 15s
Wall time: 5h 10min 7s


In [ ]:
%%time
%%capture

df_value['train_sac'],df_actions['train_sac'],df_value['test_sac'],df_actions['test_sac']=train_and_predict("sac", processed_full, ENV_PARAMS, SAC_PARAMS, total_timesteps=100000)

CPU times: user 39min 14s, sys: 32min 32s, total: 1h 11min 47s
Wall time: 2h 30min 19s


## Evaluation
Backtesting plays a key role in evaluating the performance of a trading strategy. Automated backtesting tool is preferred because it reduces the human error. We usually use the Quantopian pyfolio package to backtest our trading strategies. It is easy to use and consists of various individual plots that provide a comprehensive image of the performance of a trading strategy.

### Baseline Performance

In [31]:
# get baseline stats
print("baseline performance backtest:")
training_baseline_df = get_baseline(ticker="SPY", start=start_date,end=split_date)
evaluation_baseline_df = get_baseline(ticker="SPY", start=split_date,end=end_date)

print()
print("================Training Period Perf================")
stats = backtest_stats(training_baseline_df, value_col_name='close')
print()
print("================Evaluation Period Perf================")
stats = backtest_stats(evaluation_baseline_df, value_col_name='close')

baseline performance backtest:


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (5974, 8)


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (501, 8)

================Training Period Perf================
Annual return          0.065850
Cumulative returns     3.534991
Annual volatility      0.196868
Sharpe ratio           0.422526
Calmar ratio           0.119317
Stability              0.859077
Max drawdown          -0.551895
Omega ratio            1.083762
Sortino ratio          0.596082
Skew                        NaN
Kurtosis                    NaN
Tail ratio             0.904136
Daily value at risk   -0.024473
dtype: float64

================Evaluation Period Perf================
Annual return          0.266414
Cumulative returns     0.599300
Annual volatility      0.163490
Sharpe ratio           1.529208
Calmar ratio           1.420477
Stability              0.842894
Max drawdown          -0.187552
Omega ratio            1.348192
Sortino ratio          2.317524
Skew                        NaN
Kurtosis                    NaN
Tail ratio             0.974858
Daily value at risk   -0.019606
dtype: float6

### Evaluation Performance

In [38]:
split_date

'2023-10-01'

In [ ]:
%matplotlib inline
backtest_plot(df_value['test_a2c'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (501, 8)



/Users/minchen.wang/Documents/financial-agent/.venv/lib/python3.12/site-packages/pyfolio/plotting.py:670: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-5.218%' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  perf_stats.loc[stat, column] = str(np.round(value * 100, 3)) + "%"


Start date,2023-10-02
End date,2025-09-30
Total months,59
,Backtest
Annual return,-5.218%
Cumulative returns,-23.41%
Annual volatility,20.014%
Sharpe ratio,-0.17
Calmar ratio,-0.08
Stability,0.17
Max drawdown,-63.932%


ValueError: Incompatible indexer with Series

In [33]:
%matplotlib inline
backtest_plot(df_value['test_ppo'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_ppo'

In [34]:
%matplotlib inline
backtest_plot(df_value['test_ddpg'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_ddpg'

In [35]:
%matplotlib inline
backtest_plot(df_value['test_td3'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_td3'

In [36]:
%matplotlib inline
backtest_plot(df_value['test_sac'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_sac'